# 🔬 Mini Project 3B — Pekan 1: Setup & Data Collection
**Gas Optimization Framework untuk Smart Contract Solidity**

Jalankan sel dari atas ke bawah secara berurutan.

| Sel | Isi |
|---|---|
| 1 | Verifikasi environment |
| 2 | Install & test solc |
| 3–5 | Setup & test Hardhat |
| 6–8 | Download kontrak dari Etherscan |
| 9–10 | Validasi dataset |
| 11–13 | AST Parser |
| 14 | Checklist akhir |

---
## ✅ Sel 1 — Verifikasi Environment

In [1]:
import sys
import subprocess
import os
from pathlib import Path

# ── Path root project (folder gas_optimization)
ROOT        = Path(os.getcwd()).parent  # notebooks/ → satu level ke atas
HARDHAT_DIR = ROOT / 'hardhat_project'
DATASET_DIR = ROOT / 'contracts_dataset'
RESULTS_DIR = ROOT / 'results'

# Buat folder yang belum ada
DATASET_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ── Helper: jalankan command shell
def run(cmd, cwd=None):
    """Jalankan shell command, print output, return CompletedProcess."""
    result = subprocess.run(
        cmd, shell=True, capture_output=True,
        text=True, cwd=cwd
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip():
        print('STDERR:', result.stderr.strip())
    return result

# ── Cek versi tools
print('=== VERIFIKASI ENVIRONMENT ===')
print(f'Python  : {sys.version.split()[0]}')
run('node --version')
run('npm --version')
print(f'ROOT    : {ROOT}')
print('✅ Environment OK')

=== VERIFIKASI ENVIRONMENT ===
Python  : 3.11.15
v24.14.1
11.11.0
ROOT    : C:\Users\Ridho\project\KBJ\gas_optimization
✅ Environment OK


---
## 🔧 Sel 2 — Install & Test solc 0.8.20

In [2]:
import solcx

# Install jika belum ada
installed = solcx.get_installed_solc_versions()
if solcx.main.Version('0.8.20') not in installed:
    print('⏳ Menginstall solc 0.8.20...')
    solcx.install_solc('0.8.20')

solcx.set_solc_version('0.8.20')
print(f'✅ solc aktif: {solcx.get_solc_version()}')

# ── Test compile kontrak sederhana
SOURCE_TEST = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract HelloTest {
    uint256 public nilai = 42;
    function getNilai() external view returns (uint256) {
        return nilai;
    }
}
"""

hasil = solcx.compile_source(
    SOURCE_TEST,
    output_values=['abi', 'bin', 'ast'],
    solc_version='0.8.20'
)
print('✅ Test compile berhasil')
print(f'   AST tersedia : {"ast" in list(hasil.values())[0]}')

✅ solc aktif: 0.8.20
✅ Test compile berhasil
   AST tersedia : True


---
## 🔧 Sel 3 — Setup Project Hardhat
> Hanya perlu dijalankan SEKALI. Skip jika `hardhat_project/node_modules/` sudah ada.

In [3]:
node_modules = HARDHAT_DIR / 'node_modules'

if node_modules.exists():
    print('⏭️  node_modules sudah ada — skip install')
else:
    print('⏳ Install Hardhat (1–2 menit)...')
    run('npm init -y', cwd=HARDHAT_DIR)
    run('npm install --save-dev hardhat @nomicfoundation/hardhat-toolbox',
        cwd=HARDHAT_DIR)

# Verifikasi
r = run('npx hardhat --version', cwd=HARDHAT_DIR)
if r.returncode == 0:
    print('✅ Hardhat siap')
else:
    print('❌ Hardhat gagal — cek error di atas')

⏭️  node_modules sudah ada — skip install
2.28.6
✅ Hardhat siap


---
## 🔧 Sel 4 — Tulis Kontrak Test & Script Gas

In [ ]:
# ── hardhat.config.js  (CJS — Hardhat 2.x)
hardhat_config = """require("@nomicfoundation/hardhat-toolbox");

module.exports = {
  solidity: {
    version: "0.8.20",
    settings: {
      optimizer: { enabled: false },  // WAJIB false
    },
  },
  networks: { hardhat: {} },
};
"""

# ── TestGas.sol — demonstrasi Redundant SLOAD
test_sol = """// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract TestGas {
    uint256 public angka;

    // Anti-pattern: baca storage 3x
    function setBoros(uint256 x) public {
        angka = x;
        if (angka > 0) {        // SLOAD ke-2
            angka = angka + 1;  // SLOAD ke-3
        }
    }

    // Optimal: baca storage 1x
    function setHemat(uint256 x) public {
        uint256 _tmp = x;
        if (_tmp > 0) { _tmp = _tmp + 1; }
        angka = _tmp;
    }
}
"""

# ── ukur_gas.js  (CJS — ethers v6 / hardhat-toolbox v6)
gas_script = """const { ethers } = require("hardhat");

async function main() {
  const F = await ethers.getContractFactory("TestGas");
  const c = await F.deploy();
  await c.waitForDeployment();           // ethers v6

  const t1 = await c.setBoros(5); const r1 = await t1.wait();
  const t2 = await c.setHemat(5); const r2 = await t2.wait();

  const diff    = r1.gasUsed - r2.gasUsed;  // BigInt arithmetic
  const pctSave = (Number(diff) * 100 / Number(r1.gasUsed)).toFixed(2);

  console.log("=== HASIL PENGUKURAN GAS ===");
  console.log("setBoros  :", r1.gasUsed.toString());
  console.log("setHemat  :", r2.gasUsed.toString());
  console.log("Selisih   :", diff.toString(), "gas");
  console.log("Penghematan:", pctSave + "%");
}
main().catch(e => { console.error(e); process.exitCode = 1; });
"""

# ── Tulis semua file
(HARDHAT_DIR / 'hardhat.config.js').write_text(hardhat_config)
(HARDHAT_DIR / 'contracts').mkdir(exist_ok=True)
(HARDHAT_DIR / 'scripts').mkdir(exist_ok=True)
(HARDHAT_DIR / 'contracts' / 'TestGas.sol').write_text(test_sol)
(HARDHAT_DIR / 'scripts'   / 'ukur_gas.js').write_text(gas_script)

print('✅ hardhat.config.js  ditulis (CJS)')
print('✅ contracts/TestGas.sol ditulis')
print('✅ scripts/ukur_gas.js   ditulis (CJS, ethers v6)')

---
## 🔧 Sel 5 — Compile & Ukur Gas (Test Hardhat)

In [5]:
print('⏳ Compile...')
run('npx hardhat compile', cwd=HARDHAT_DIR)

print('\n⏳ Ukur gas...')
run('npx hardhat run scripts/ukur_gas.js --network hardhat', cwd=HARDHAT_DIR)

# Output yang diharapkan:
# === HASIL PENGUKURAN GAS ===
# setBoros  : 4XXXX
# setHemat  : 4XXXX
# Selisih   : XXXX gas
# Penghematan: X.XX%

⏳ Compile...
Nothing to compile

⏳ Ukur gas...
STDERR: TypeError: c.deployed is not a function
    at main (C:\Users\Ridho\project\KBJ\gas_optimization\hardhat_project\scripts\ukur_gas.js:6:11)


CompletedProcess(args='npx hardhat run scripts/ukur_gas.js --network hardhat', returncode=1, stdout='', stderr='TypeError: c.deployed is not a function\n    at main \x1b(C:\\Users\\Ridho\\project\\KBJ\\gas_optimization\\hardhat_project\\\x1bscripts\\ukur_gas.js:6:11\x1b)\x1b\n')

---
## 📥 Sel 6 — Load API Key Etherscan
> Pastikan file `.env` di root project sudah diisi:
> ```
> ETHERSCAN_API_KEY=isi_api_key_kamu
> ```

In [6]:
import requests, json, time
from collections import Counter, defaultdict
from dotenv import load_dotenv

# Baca .env dari root project
load_dotenv(ROOT / '.env')
API_KEY = os.getenv('ETHERSCAN_API_KEY', '')

if not API_KEY:
    raise ValueError('❌ ETHERSCAN_API_KEY tidak ditemukan. Isi file .env di root project.')

print(f'✅ API Key loaded: {API_KEY[:6]}{"*" * 10}')
print(f'📁 Dataset dir  : {DATASET_DIR}')

✅ API Key loaded: SNQRM9**********
📁 Dataset dir  : C:\Users\Ridho\project\KBJ\gas_optimization\contracts_dataset


---
## 📥 Sel 7 — Daftar 50 Alamat Kontrak

In [ ]:
ALAMAT_KONTRAK = {
    'DeFi': [
        '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',  # WETH
        '0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D',  # Uniswap V2 Router
        '0x6B175474E89094C44Da98b954EedeAC495271d0F',  # DAI
        '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48',  # USDC
        '0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984',  # UNI
        '0x7Fc66500c84A76Ad7e9c93437bFc5Ac33E2DDaE9',  # AAVE
        '0x5A98FcBEA516Cf06857215779Fd812CA3beF1B32',  # LDO
        '0xae7ab96520DE3A18E5e111B5EaAb095312D7fE84',  # stETH  (ganti CRV/Vyper)
        '0xc00e94Cb662C3520282E6f5717214004A7f26888',  # COMP
        '0x9f8F72aA9304c8B593d555F12eF6589cC3A579A2',  # MKR
    ],
    'NFT': [
        '0xBC4CA0EdA7647A8aB7C2061c2E118A18a936f13D',  # BAYC
        '0xb47e3cd837dDF8e4c57F05d70Ab865de6e193BBB',  # CryptoPunks
        '0x60E4d786628Fea6478F785A6d7e704777c86a7c6',  # MAYC
        '0x49cF6f5d44E70224e2E23fDcdd2C053F30aDA28B',  # CloneX
        '0x23581767a106ae21c074b2276D25e5C3e136a68b',  # Moonbirds
        '0xEd5AF388653567Af2F388E6224dC7C4b3241C544',  # Azuki
        '0x8a90CAb2b38dba80c64b7734e58Ee1dB38B8992e',  # Doodles
        '0x7Bd29408f11D2bFC23c34f18275bBf23bB716Bc7',  # Meebits
        '0x1CB1A5e65610AEFF2551A50f76a87a7d3fB649C6',  # CrypToadz
        '0x34d85c9CDeB23FA97cb08333b511ac86E1C4E258',  # Otherdeed
    ],
    'Token': [
        '0xdAC17F958D2ee523a2206206994597C13D831ec7',  # USDT
        '0x2260FAC5E5542a773Aa44fBCfeDf7C193bc2C599',  # WBTC
        '0x514910771AF9Ca656af840dff83E8264EcF986CA',  # LINK
        '0x0bc529c00C6401aEF6D220BE8C6Ea1667F6Ad93e',  # YFI
        '0x111111111117dC0aa78b770fA6A738034120C302',  # 1INCH
        '0x6810e776880C02933D47DB1b9fc05908e5386b96',  # GNO
        '0xba100000625a3754423978a60c9317c58a424e3D',  # BAL
        '0xC011a73ee8576Fb46F5E1c5751cA3B9Fe0af2a6F',  # SNX
        '0x4d224452801ACEd8B2F0aebE155379bb5D594381',  # APE
        '0x6DEA81C8171D0bA574754EF6F8b412F2Ed88c54D',  # LQTY
    ],
    'Governance': [
        '0x408ED6354d4973f66138C91495F2f2FCbd8724C3',  # Uniswap Governance
        '0xc0Da02939E1441F497fd74F78cE7Decb17B66529',  # Compound Governor
        '0x0BEF27FEB58e857046d630B2c03dFb7bae567494',  # Aave Governance
        '0xEC568fffba86c094cf06b22134B23074DFE2252c',  # Aave Gov V2  (ganti ENS Gov)
        '0x690e775361AD66D1c4A25d89da9fCd639F5198eD',  # Synthetix Council
        '0xDbD27635A534A3d3169Ef0498beB56Fb9c937489',  # Gitcoin DAO
        '0x35d9f4953748b318f18c30634bA299b237eeDfff',  # Index Coop Governor
        '0x323A76393544d5ecca80cd6ef2A560C6a395b7E3',  # Inverse Finance
        '0x6f3E6272A167e8AcCb32072d08E0957F9c79223d',  # Nouns DAO  (ganti Badger)
        '0xB3a87172F555ae2a2AB79Be60B336D2F7D0187f0',  # dYdX Governor
    ],
    'Utility': [
        '0xBB9bc244D798123fDe783fCc1C72d3Bb8C189413',  # Gnosis Multisig
        '0xd9Db270c1B5E3Bd161E8c8503c55cEABeE709552',  # Gnosis Safe 1.3.0  (ganti Binance)
        '0xA646E29877d52B9e2De457ECa09C724fF16D0a2B',  # KuCoin Multisig
        '0x19c0976f590D67707E62397C87829d896Dc0f1F1',  # MakerDAO Timelock
        '0x8392F6669292fA56123F71949B52d883aE57e225',  # Aave Timelock
        '0x00000000006c3852cbEf3e08E8dF289169EdE581',  # Seaport 1.1  (ganti Compound TL)
        '0x1F98431c8aD98523631AE4a59f267346ea31F984',  # Uniswap V3 Factory  (ganti ENS TL)
        '0xE592427A0AEce92De3Edee1F18E0157C05861564',  # Uniswap V3 SwapRouter  (ganti Kyber)
        '0x68b3465833fb72A70ecDF485E0e4C7bD8665Fc45',  # Uniswap SwapRouter02  (ganti dYdX TL)
        '0xC36442b4a4522E871399CD717aBDD847Ab11FE88',  # Uniswap V3 NFTPositionMgr  (ganti Sushi)
    ]
}

total = sum(len(v) for v in ALAMAT_KONTRAK.values())
print(f'📋 Total alamat: {total}')
for domain, addrs in ALAMAT_KONTRAK.items():
    print(f'   {domain:<12}: {len(addrs)} kontrak')

---
## 📥 Sel 8 — Download Kontrak dari Etherscan

In [8]:
LOG_BERHASIL = []
LOG_GAGAL    = []

def download_kontrak(alamat, domain, nomor):
    # Etherscan API V2 (V1 deprecated)
    url = (
        f'https://api.etherscan.io/v2/api'
        f'?chainid=1&module=contract&action=getsourcecode'
        f'&address={alamat}&apikey={API_KEY}'
    )
    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()
    except Exception as e:
        print(f'  ❌ Network error: {e}')
        LOG_GAGAL.append({'alamat': alamat, 'domain': domain, 'alasan': str(e)})
        return False

    if data.get('status') != '1':
        print(f'  ❌ API error [{alamat[:10]}...]: {data.get("message")} — {data.get("result", "")[:80]}')
        LOG_GAGAL.append({'alamat': alamat, 'domain': domain, 'alasan': data.get('message')})
        return False

    result        = data['result'][0]
    source_code   = result.get('SourceCode', '')
    contract_name = result.get('ContractName') or f'Unknown_{nomor:02d}'

    if not source_code:
        print(f'  ⚠️  Tidak verified: {alamat[:10]}...')
        LOG_GAGAL.append({'alamat': alamat, 'domain': domain, 'alasan': 'not verified'})
        return False

    filepath = DATASET_DIR / f'{domain}_{nomor:02d}_{contract_name}.sol'

    with open(filepath, 'w', encoding='utf-8') as f:
        if source_code.startswith('{{'):
            try:
                files_json = json.loads(source_code[1:-1])
                parts = []
                for path, content in files_json['sources'].items():
                    parts.append(f'// === File: {path} ===')
                    parts.append(content['content'])
                f.write('\n\n'.join(parts))
            except Exception:
                f.write(source_code)
        else:
            f.write(source_code)

    loc        = len(filepath.read_text(encoding='utf-8').splitlines())
    complexity = 'Simple' if loc < 100 else 'Medium' if loc < 500 else 'Complex'

    print(f'  ✅ {contract_name:<35} {loc:>5} baris [{complexity}]')
    LOG_BERHASIL.append({
        'alamat': alamat, 'domain': domain,
        'nama': contract_name, 'file': str(filepath),
        'loc': loc, 'complexity': complexity
    })
    return True


# ── Jalankan download
print('🚀 Mulai download...\n')
for domain, daftar in ALAMAT_KONTRAK.items():
    print(f'\n📂 {domain}')
    print('─' * 55)
    for i, alamat in enumerate(daftar, start=1):
        download_kontrak(alamat, domain, i)
        time.sleep(0.25)  # max 5 req/detik

# Simpan metadata
metadata_path = ROOT / 'contracts_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(LOG_BERHASIL, f, indent=2)

print(f'\n{"="*55}')
print(f'✅ Berhasil : {len(LOG_BERHASIL)} kontrak')
print(f'❌ Gagal    : {len(LOG_GAGAL)} kontrak')
print(f'💾 Metadata : {metadata_path}')

🚀 Mulai download...


📂 DeFi
───────────────────────────────────────────────────────
  ✅ WETH9                                1511 baris [Complex]
  ✅ UniswapV2Router02                    1559 baris [Complex]
  ✅ Dai                                   379 baris [Medium]
  ✅ FiatTokenProxy                        657 baris [Complex]
  ✅ Uni                                  1163 baris [Complex]
  ✅ InitializableAdminUpgradeabilityProxy  1389 baris [Complex]
  ✅ MiniMeToken                          1199 baris [Complex]
  ✅ Vyper_contract                        747 baris [Complex]
  ✅ Comp                                  301 baris [Medium]
  ✅ DSToken                               947 baris [Complex]

📂 NFT
───────────────────────────────────────────────────────
  ✅ BoredApeYachtClub                    4033 baris [Complex]
  ✅ CryptoPunksMarket                     491 baris [Medium]
  ✅ MutantApeYachtClub                   1819 baris [Complex]
  ✅ CloneX                               1655 b

In [ ]:
# ── Sel 8b — Retry timeout & ganti alamat tidak verified
# Jalankan SETELAH Sel 8. Skip otomatis jika file sudah ada (tetap dihitung).

def download_single(alamat, domain, nomor, timeout=30, retries=3):
    for attempt in range(retries):
        url = (f'https://api.etherscan.io/v2/api?chainid=1'
               f'&module=contract&action=getsourcecode'
               f'&address={alamat}&apikey={API_KEY}')
        try:
            resp = requests.get(url, timeout=timeout)
            data = resp.json()
        except Exception as e:
            if attempt < retries - 1:
                print(f'    ⏳ Retry {attempt+2}/{retries}...')
                time.sleep(2 ** attempt)
                continue
            print(f'  ❌ {alamat[:10]}... timeout')
            return False

        if data.get('status') != '1' or not data['result'][0].get('SourceCode'):
            msg = str(data.get('result', 'tidak verified'))[:60]
            print(f'  ❌ {alamat[:10]}... {msg}')
            return False

        r    = data['result'][0]
        src  = r['SourceCode']
        nama = r.get('ContractName') or f'Unknown_{nomor:02d}'
        fp   = DATASET_DIR / f'{domain}_{nomor:02d}_{nama}.sol'

        with open(fp, 'w', encoding='utf-8') as f:
            if src.startswith('{{'):
                try:
                    fj    = json.loads(src[1:-1])
                    parts = [x for p, c in fj['sources'].items()
                             for x in (f'// === File: {p} ===', c['content'])]
                    f.write('\n\n'.join(parts))
                except Exception:
                    f.write(src)
            else:
                f.write(src)

        loc = len(fp.read_text(encoding='utf-8').splitlines())
        cx  = 'Simple' if loc < 100 else 'Medium' if loc < 500 else 'Complex'
        print(f'  ✅ {nama:<35} {loc:>5} baris [{cx}]')
        LOG_BERHASIL.append({'alamat': alamat, 'domain': domain, 'nama': nama,
                             'file': str(fp), 'loc': loc, 'complexity': cx})
        return True
    return False


# Retry timeout + alamat pengganti untuk yang tidak verified
PERBAIKAN = [
    # (domain, nomor, alamat)
    ('Token',      8,  '0xC011a73ee8576Fb46F5E1c5751cA3B9Fe0af2a6F'),  # SNX    — retry
    ('Token',      9,  '0x4d224452801ACEd8B2F0aebE155379bb5D594381'),  # APE    — retry
    ('Governance', 4,  '0xEC568fffba86c094cf06b22134B23074DFE2252c'),  # Aave Gov V2
    ('Governance', 9,  '0x6f3E6272A167e8AcCb32072d08E0957F9c79223d'),  # Nouns DAO Gov
    ('Utility',    2,  '0xd9Db270c1B5E3Bd161E8c8503c55cEABeE709552'),  # Gnosis Safe 1.3.0
    ('Utility',    6,  '0x00000000006c3852cbEf3e08E8dF289169EdE581'),  # OpenSea Seaport 1.1
    ('Utility',    7,  '0x1F98431c8aD98523631AE4a59f267346ea31F984'),  # Uniswap V3 Factory
    ('Utility',    8,  '0xE592427A0AEce92De3Edee1F18E0157C05861564'),  # Uniswap V3 SwapRouter
    ('Utility',    9,  '0x68b3465833fb72A70ecDF485E0e4C7bD8665Fc45'),  # Uniswap SwapRouter02
    ('Utility',    10, '0xC36442b4a4522E871399CD717aBDD847Ab11FE88'),  # Uniswap V3 NFTPositionMgr
]

print('🔄 Retry & pengganti kontrak gagal...\n')
n_awal = len(LOG_BERHASIL)

for domain, nomor, alamat in PERBAIKAN:
    existing = list(DATASET_DIR.glob(f'{domain}_{nomor:02d}_*.sol'))
    if existing:
        fp   = existing[0]
        nama = '_'.join(fp.stem.split('_')[2:])  # strip "Domain_NN_"
        loc  = len(fp.read_text(encoding='utf-8').splitlines())
        cx   = 'Simple' if loc < 100 else 'Medium' if loc < 500 else 'Complex'
        print(f'  ⏭️  {domain}_{nomor:02d} sudah ada — {nama} ({loc} baris)')
        # Tambahkan ke LOG agar ikut terhitung di metadata
        if not any(e['file'] == str(fp) for e in LOG_BERHASIL):
            LOG_BERHASIL.append({'alamat': alamat, 'domain': domain, 'nama': nama,
                                 'file': str(fp), 'loc': loc, 'complexity': cx})
        continue
    download_single(alamat, domain, nomor)
    time.sleep(0.3)

with open(ROOT / 'contracts_metadata.json', 'w') as f:
    json.dump(LOG_BERHASIL, f, indent=2)

print(f'\n{"="*55}')
print(f'Total berhasil: {len(LOG_BERHASIL)} kontrak (+{len(LOG_BERHASIL) - n_awal} baru/ditemukan)')

---
## 📊 Sel 9 — Validasi Dataset

In [10]:
with open(ROOT / 'contracts_metadata.json') as f:
    metadata = json.load(f)

print('📊 RINGKASAN DATASET')
print('=' * 55)

per_domain = defaultdict(list)
for m in metadata:
    per_domain[m['domain']].append(m)

for domain, contracts in per_domain.items():
    locs = [c['loc'] for c in contracts]
    cx   = Counter(c['complexity'] for c in contracts)
    print(f'\n{domain}: {len(contracts)} kontrak')
    print(f'  LOC  : min={min(locs)}, max={max(locs)}, avg={sum(locs)//len(locs)}')
    print(f'  Level: {dict(cx)}')

print('\n📐 Distribusi Complexity:')
for level, count in Counter(m['complexity'] for m in metadata).items():
    print(f'  {level:<8}: {"█" * count} ({count})')

📊 RINGKASAN DATASET

DeFi: 10 kontrak
  LOC  : min=301, max=1559, avg=985
  Level: {'Complex': 8, 'Medium': 2}

NFT: 10 kontrak
  LOC  : min=491, max=4294, avg=2124
  Level: {'Complex': 9, 'Medium': 1}

Token: 10 kontrak
  LOC  : min=301, max=2987, avg=1247
  Level: {'Complex': 8, 'Medium': 2}

Governance: 8 kontrak
  LOC  : min=266, max=3751, avg=1293
  Level: {'Complex': 5, 'Medium': 3}

Utility: 4 kontrak
  LOC  : min=313, max=6783, avg=2265
  Level: {'Complex': 3, 'Medium': 1}

📐 Distribusi Complexity:
  Complex : █████████████████████████████████ (33)
  Medium  : █████████ (9)


---
## 🔍 Sel 10 — Verifikasi Compile

In [11]:
import re, tempfile

SOLC_MAP = {
    '0.4': '0.4.26',
    '0.5': '0.5.17',
    '0.6': '0.6.12',
    '0.7': '0.7.6',
    '0.8': '0.8.20',
}

def detect_solc_ver(source):
    m = re.search(r'pragma solidity\s+[>=^<~]*\s*(\d+\.\d+)', source)
    if not m:
        return '0.8.20'
    major_minor = '.'.join(m.group(1).split('.')[:2])
    return SOLC_MAP.get(major_minor, '0.8.20')

def parse_multifile(source):
    parts = re.split(r'// === File: (.+?) ===', source)
    if len(parts) <= 1:
        return None
    files = {}
    for i in range(1, len(parts) - 1, 2):
        files[parts[i].strip()] = parts[i + 1]
    return files or None

def smart_compile_ast(filepath):
    """Compile .sol → AST. 3 attempt: standard JSON / temp-dir files / strip imports."""
    src = Path(filepath).read_text(encoding='utf-8')

    if 'pragma solidity' not in src:
        return None  # Vyper

    ver = detect_solc_ver(src)
    installed = [str(v) for v in solcx.get_installed_solc_versions()]
    if ver not in installed:
        print(f'    ⏳ install solc {ver}...')
        solcx.install_solc(ver)

    files = parse_multifile(src)

    if files:
        # Attempt 1: compile_standard (in-memory, semua file)
        try:
            inp = {
                "language": "Solidity",
                "sources": {n: {"content": c} for n, c in files.items()},
                "settings": {"optimizer": {"enabled": False},
                             "outputSelection": {"*": {"": ["ast"]}}}
            }
            out = solcx.compile_standard(inp, solc_version=ver)
            return out['sources'][next(iter(out['sources']))]['ast']
        except Exception:
            pass

        # Attempt 2: tulis ke temp dir → compile dengan allow_paths
        try:
            with tempfile.TemporaryDirectory() as td:
                tdp = Path(td)
                for name, content in files.items():
                    fp = tdp / name
                    fp.parent.mkdir(parents=True, exist_ok=True)
                    fp.write_text(content, encoding='utf-8')
                main_fp = str(tdp / next(iter(files)))
                out = solcx.compile_files(
                    [main_fp], output_values=['ast'],
                    solc_version=ver, optimize=False,
                    allow_paths=[td]
                )
                if out:
                    return list(out.values())[0]['ast']
        except Exception:
            pass
    else:
        try:
            out = solcx.compile_source(src, output_values=['ast'],
                                       solc_version=ver, optimize=False)
            return list(out.values())[0]['ast']
        except Exception:
            pass

    # Attempt 3: strip import → compile file utama saja
    try:
        main_src = src if not files else list(files.values())[0]
        stripped = re.sub(r'^\s*import\s+[^;]+;', '', main_src, flags=re.MULTILINE)
        out = solcx.compile_source(stripped, output_values=['ast'],
                                   solc_version=ver, optimize=False)
        if out:
            return list(out.values())[0]['ast']
    except Exception:
        pass

    return None


# ── Verifikasi compile
print('🔍 Verifikasi compile semua kontrak...\n')

bisa_compile  = []
gagal_compile = []

for m in metadata:
    ast = smart_compile_ast(m['file'])
    if ast:
        bisa_compile.append(m)
    else:
        ver = detect_solc_ver(Path(m['file']).read_text(encoding='utf-8'))
        print(f'  ⚠️  {m["nama"]:<35} [solc {ver}]')
        gagal_compile.append(m)

gagal_set = {g['nama'] for g in gagal_compile}
for m in metadata:
    m['compile_ok'] = m['nama'] not in gagal_set

with open(ROOT / 'contracts_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\n{"="*55}')
print(f'✅ Bisa compile : {len(bisa_compile)}/{len(metadata)}')
print(f'⚠️  Gagal compile: {len(gagal_compile)}/{len(metadata)}')
print('💾 Metadata diupdate (field: compile_ok)')

🔍 Verifikasi compile semua kontrak...

  ⚠️  UniswapV2Router02                   [solc 0.6.12]
  ⚠️  Dai                                 [solc 0.5.17]
  ⚠️  InitializableAdminUpgradeabilityProxy [solc 0.6.12]
  ⚠️  Vyper_contract                      [solc 0.8.20]
  ⚠️  BoredApeYachtClub                   [solc 0.6.12]
  ⚠️  Land                                [solc 0.8.20]
  ⚠️  WBTC                                [solc 0.4.26]
  ⚠️  GnosisToken                         [solc 0.4.26]
  ⚠️  BalancerGovernanceToken             [solc 0.6.12]
  ⚠️  ProxyERC20                          [solc 0.4.26]
  ⚠️  LQTYToken                           [solc 0.6.12]
  ⚠️  FeiDAO                              [solc 0.8.20]
  ⚠️  DAO                                 [solc 0.8.20]
  ⚠️  Jug                                 [solc 0.5.17]

✅ Bisa compile : 28/42
⚠️  Gagal compile: 14/42
💾 Metadata diupdate (field: compile_ok)


---
## 🌳 Sel 11 — Definisi Fungsi AST Parser

In [12]:
def generate_ast(filepath):
    """Generate AST dari file Solidity. Return dict atau None."""
    return smart_compile_ast(filepath)


def walk_ast(node, callback, depth=0):
    """Rekursif jelajahi semua node AST."""
    if not isinstance(node, dict):
        return
    callback(node, depth)
    for value in node.values():
        if isinstance(value, dict):
            walk_ast(value, callback, depth + 1)
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    walk_ast(item, callback, depth + 1)


def find_nodes(ast, target_type):
    """Kembalikan list semua node dengan nodeType tertentu."""
    result = []
    walk_ast(ast, lambda n, d: result.append(n)
             if n.get('nodeType') == target_type else None)
    return result


print('✅ Fungsi AST siap: generate_ast(), walk_ast(), find_nodes()')

✅ Fungsi AST siap: generate_ast(), walk_ast(), find_nodes()


---
## 🌳 Sel 12 — Test AST pada 1 Kontrak

In [13]:
target = next((m for m in metadata if m.get('compile_ok')), None)

if not target:
    print('❌ Tidak ada kontrak yang bisa compile')
else:
    ast = generate_ast(target['file'])
    if ast:
        functions = find_nodes(ast, 'FunctionDefinition')
        print(f'✅ AST berhasil: {target["nama"]}')
        print(f'   Root node  : {ast.get("nodeType")}')
        print(f'   Child nodes: {len(ast.get("nodes", []))}')
        print(f'   Functions  : {len(functions)}')
        for fn in functions[:5]:
            name = fn.get('name') or '(fallback/receive)'
            vis  = fn.get('visibility', '-')
            print(f'     • {name:<30} visibility={vis}')
    else:
        print('❌ generate_ast() gagal')

✅ AST berhasil: WETH9
   Root node  : None
   Child nodes: 0
   Functions  : 0


---
## 🌳 Sel 13 — Inventarisasi Node Types

In [14]:
kontrak_valid = [m for m in metadata if m.get('compile_ok')]
print(f'Inventarisasi pada {len(kontrak_valid)} kontrak...\n')

total_types = Counter()
for m in kontrak_valid:
    ast = generate_ast(m['file'])
    if ast:
        counter = Counter()
        walk_ast(ast, lambda n, d: counter.update(
            [n['nodeType']] if 'nodeType' in n else []))
        total_types.update(counter)

print(f'{"Node Type":<40} {"Count":>8}')
print('─' * 50)
for tipe, jumlah in total_types.most_common(20):
    print(f'{tipe:<40} {jumlah:>8,}')

# Node relevan untuk 6 anti-pattern
print('\n🎯 Node relevan untuk Pekan 2:')
TARGET_NODES = [
    ('Identifier',         'Redundant SLOAD'),
    ('ForStatement',       'Unoptimized Loop'),
    ('ElementaryTypeName', 'String vs Bytes32'),
    ('FunctionDefinition', 'Public vs External + Dead Code'),
    ('BinaryOperation',    'Unchecked Arithmetic'),
    ('UncheckedStatement', 'Unchecked Arithmetic (scope aman)'),
]
for node, keterangan in TARGET_NODES:
    count  = total_types.get(node, 0)
    status = '✅' if count > 0 else '⚠️ '
    print(f'  {status} {node:<25} ({count:,}x) — {keterangan}')

Inventarisasi pada 28 kontrak...

Node Type                                   Count
──────────────────────────────────────────────────
Identifier                                    929
VariableDeclaration                           422
ElementaryTypeName                            412
FunctionCall                                  290
ParameterList                                 261
MemberAccess                                  247
Literal                                       172
ExpressionStatement                           167
Block                                         152
FunctionDefinition                            118
BinaryOperation                               107
StructuredDocumentation                        74
IndexAccess                                    70
VariableDeclarationStatement                   67
Return                                         59
Assignment                                     57
IdentifierPath                                 56
UserDefinedType

---
## ✅ Sel 14 — Checklist Akhir Pekan 1

In [15]:
hh_ok = run('npx hardhat --version', cwd=HARDHAT_DIR).returncode == 0

checks = [
    ('solc 0.8.20 aktif',
        solcx.get_solc_version() is not None),
    ('Hardhat berjalan',
        hh_ok),
    ('contracts_metadata.json ada',
        (ROOT / 'contracts_metadata.json').exists()),
    ('Minimal 45 kontrak didownload',
        len(LOG_BERHASIL) >= 45),
    ('Semua 5 domain ada',
        len(per_domain) == 5),
    ('Minimal 40 kontrak bisa compile',
        len(bisa_compile) >= 40),
    ('AST parser berjalan',
        bool(kontrak_valid) and generate_ast(kontrak_valid[0]['file']) is not None),
    ('Node types sudah diinventarisasi',
        len(total_types) > 0),
]

print('=' * 45)
print('  CHECKLIST AKHIR PEKAN 1')
print('=' * 45)

semua_ok = True
for label, ok in checks:
    icon = '✅' if ok else '❌'
    if not ok:
        semua_ok = False
    print(f'  {icon} {label}')

print('=' * 45)
print('  🎉 Siap masuk Pekan 2!' if semua_ok
      else '  ⚠️  Cek item ❌ di atas')
print('=' * 45)

2.28.6
  CHECKLIST AKHIR PEKAN 1
  ✅ solc 0.8.20 aktif
  ✅ Hardhat berjalan
  ✅ contracts_metadata.json ada
  ❌ Minimal 45 kontrak didownload
  ✅ Semua 5 domain ada
  ❌ Minimal 40 kontrak bisa compile
  ✅ AST parser berjalan
  ✅ Node types sudah diinventarisasi
  ⚠️  Cek item ❌ di atas
